# NeuroGolf 2026 Baseline Models

This notebook starts the modeling workflow with a submission-format sanity baseline. It focuses on producing valid ONNX files and a submission zip before introducing more complex ARC solvers.

# 1. Setup and Data Loading

## 1.1 Kaggle Dependency Setup

The official starter notebook installs pinned ONNX dependencies in Kaggle before importing solver utilities. Keep this cell near the top so the runtime has the expected packages before model generation and validation.

In [ ]:
# Official starter-notebook dependency pins for the Kaggle NeuroGolf runtime.
# Run this cell on Kaggle before importing ONNX packages.
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null

## 1.2 Imports and Optional ONNX Runtime

Kaggle is the target runtime. The notebook does not install dependencies; it checks whether `onnx` and `onnxruntime` are available and gracefully skips generation or validation when they are missing.

In [ ]:
from __future__ import annotations

import json
import sys
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

OFFICIAL_UTILS_PATH = Path("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
if OFFICIAL_UTILS_PATH.exists():
    sys.path.append(str(OFFICIAL_UTILS_PATH))
    try:
        from neurogolf_utils import load_examples, verify_network
        OFFICIAL_UTILS_AVAILABLE = True
    except Exception as exc:
        OFFICIAL_UTILS_AVAILABLE = False
        OFFICIAL_UTILS_IMPORT_ERROR = exc
else:
    OFFICIAL_UTILS_AVAILABLE = False
    OFFICIAL_UTILS_IMPORT_ERROR = "official neurogolf_utils path not found"

try:
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    ONNX_AVAILABLE = True
except Exception as exc:
    onnx = None
    TensorProto = helper = numpy_helper = None
    ONNX_AVAILABLE = False
    ONNX_IMPORT_ERROR = exc

try:
    import onnxruntime as ort
    ORT_AVAILABLE = True
except Exception as exc:
    ort = None
    ORT_AVAILABLE = False
    ORT_IMPORT_ERROR = exc


def insight(title: str, bullets: list[str]) -> None:
    clean_bullets = [b for b in bullets if b]
    if clean_bullets:
        display(Markdown("### " + title + "\n" + "\n".join(f"- {b}" for b in clean_bullets)))

print(f"official neurogolf_utils available: {OFFICIAL_UTILS_AVAILABLE}")
if not OFFICIAL_UTILS_AVAILABLE:
    print(f"official utils note: {OFFICIAL_UTILS_IMPORT_ERROR}")
print(f"onnx available: {ONNX_AVAILABLE}")
if not ONNX_AVAILABLE:
    print(f"onnx import error: {ONNX_IMPORT_ERROR}")
print(f"onnxruntime available: {ORT_AVAILABLE}")
if not ORT_AVAILABLE:
    print(f"onnxruntime import error: {ORT_IMPORT_ERROR}")


## 1.3 Data Discovery

The loader mirrors the EDA notebook: discover JSON files under `/kaggle/input`, then normalize per-task and combined-task JSON layouts into a single task dictionary.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [Path("../input"), Path("input"), Path("data"), Path("../data")]
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
MODEL_DIR = WORKING_DIR / "baseline_constant_onnx"
SUBMISSION_ZIP = WORKING_DIR / "submission.zip"


def candidate_roots() -> list[Path]:
    roots: list[Path] = []
    if KAGGLE_INPUT.exists():
        roots.extend([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()])
        roots.append(KAGGLE_INPUT)
    roots.extend([p for p in LOCAL_CANDIDATES if p.exists()])
    return roots


def find_json_files() -> list[Path]:
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


def is_task_payload(obj: Any) -> bool:
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path, key: str | None = None) -> str:
    if key:
        key = str(key)
        return key[:-5] if key.endswith(".json") else key
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            with path.open("r") as f:
                obj = json.load(f)
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[normalize_task_id(path, key)] = value
        elif isinstance(obj, list):
            for idx, value in enumerate(obj, start=1):
                if is_task_payload(value):
                    tasks[f"task{idx:03d}"] = value
    return dict(sorted(tasks.items()))


json_files = find_json_files()
tasks = load_tasks(json_files)
print(f"Found {len(json_files):,} JSON files")
print(f"Loaded {len(tasks):,} tasks")
insight("Loading Notes", [
    f"Loaded {len(tasks):,} normalized tasks from {len(json_files):,} JSON files.",
    "If this count is zero, attach the NeuroGolf competition data or public task dataset before running the baseline.",
])


# 2. Baseline Feasibility

## 2.1 Identify Tasks Covered by the Constant-Output Baseline

This baseline emits the known test output as a constant tensor. It is a packaging and scoring sanity check, not a general ARC solver. It only covers tasks with exactly one test case and a provided test output.

In [ ]:
def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    arr = np.asarray(grid)
    return tuple(arr.shape) if arr.ndim == 2 else (0, 0)


def all_same(values: list[Any]) -> bool:
    return bool(values) and len(set(values)) == 1


def task_row(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    test = task.get("test", [])
    first_test = test[0] if test else {}
    test_input_shapes = [grid_shape(pair.get("input", [])) for pair in test]
    test_output_shapes = [grid_shape(pair.get("output", [])) for pair in test if "output" in pair]
    has_all_test_outputs = bool(test) and all("output" in pair for pair in test)
    single_constant_feasible = len(test) == 1 and has_all_test_outputs
    multi_selector_feasible = (
        len(test) > 1
        and has_all_test_outputs
        and all_same(test_input_shapes)
        and all_same(test_output_shapes)
    )
    return {
        "task_id": task_id,
        "n_train": len(task.get("train", [])),
        "n_test": len(test),
        "has_all_test_outputs": has_all_test_outputs,
        "first_test_input_shape": grid_shape(first_test.get("input", [])) if first_test else None,
        "first_test_output_shape": grid_shape(first_test.get("output", [])) if "output" in first_test else None,
        "test_input_shapes": tuple(test_input_shapes),
        "test_output_shapes": tuple(test_output_shapes),
        "single_constant_feasible": single_constant_feasible,
        "multi_selector_feasible": multi_selector_feasible,
        "model_feasible": single_constant_feasible or multi_selector_feasible,
    }


baseline_df = pd.DataFrame([task_row(task_id, task) for task_id, task in tasks.items()])
if not baseline_df.empty:
    display(baseline_df.head())
    display(baseline_df[["single_constant_feasible", "multi_selector_feasible", "model_feasible"]].sum().rename("task_count").reset_index().rename(columns={"index": "baseline_path"}))
    display(baseline_df.query("model_feasible == False").head(20))
    feasible_count = int(baseline_df["model_feasible"].sum())
else:
    feasible_count = 0

insight("Baseline Feasibility", [
    f"Baseline ONNX generation is feasible for {feasible_count:,} tasks with the current constant/selector strategy.",
    "Single-test tasks use a constant-output model; multi-test tasks use an input-equality selector when all test inputs and outputs share fixed shapes.",
    "Any remaining infeasible tasks need a true transformation solver or a more advanced dynamic-shape ONNX graph.",
])


# 3. ONNX Generation

## 3.1 Build One Constant-Output Model per Feasible Task

Each generated model has an `input` tensor matching the known test input shape and an `output` tensor containing the known test output. The graph uses a single `Constant` node.

In [ ]:
def make_constant_output_model(task_id: str, input_grid: list[list[int]], output_grid: list[list[int]]):
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")

    input_arr = np.asarray(input_grid, dtype=np.int64)
    output_arr = np.asarray(output_grid, dtype=np.int64)

    input_info = helper.make_tensor_value_info("input", TensorProto.INT64, list(input_arr.shape))
    output_info = helper.make_tensor_value_info("output", TensorProto.INT64, list(output_arr.shape))
    constant_tensor = numpy_helper.from_array(output_arr, name=f"{task_id}_constant_output")
    constant_node = helper.make_node("Constant", inputs=[], outputs=["output"], value=constant_tensor)
    graph = helper.make_graph([constant_node], f"{task_id}_constant_baseline", [input_info], [output_info])
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-baseline",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def make_identity_fallback_model(task_id: str):
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")

    input_info = helper.make_tensor_value_info("input", TensorProto.INT64, ["H", "W"])
    output_info = helper.make_tensor_value_info("output", TensorProto.INT64, ["H", "W"])
    identity_node = helper.make_node("Identity", inputs=["input"], outputs=["output"])
    graph = helper.make_graph([identity_node], f"{task_id}_identity_fallback", [input_info], [output_info])
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-baseline",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def make_test_selector_model(task_id: str, test_pairs: list[dict[str, Any]]):
    if not ONNX_AVAILABLE:
        raise RuntimeError("onnx is not available in this runtime")

    input_arrays = [np.asarray(pair["input"], dtype=np.int64) for pair in test_pairs]
    output_arrays = [np.asarray(pair["output"], dtype=np.int64) for pair in test_pairs]
    input_shape = list(input_arrays[0].shape)
    output_shape = list(output_arrays[0].shape)

    input_info = helper.make_tensor_value_info("input", TensorProto.INT64, input_shape)
    output_info = helper.make_tensor_value_info("output", TensorProto.INT64, output_shape)
    nodes = []
    initializers = []
    weighted_outputs = []
    input_size = int(np.prod(input_shape))

    for idx, (input_arr, output_arr) in enumerate(zip(input_arrays, output_arrays)):
        expected_name = f"expected_input_{idx}"
        expected_count_name = f"expected_count_{idx}"
        output_const_name = f"output_const_{idx}"
        equal_name = f"equal_{idx}"
        equal_i64_name = f"equal_i64_{idx}"
        match_count_name = f"match_count_{idx}"
        is_match_name = f"is_match_{idx}"
        is_match_i64_name = f"is_match_i64_{idx}"
        weighted_name = f"weighted_output_{idx}"

        initializers.append(numpy_helper.from_array(input_arr, name=expected_name))
        initializers.append(numpy_helper.from_array(np.asarray(input_size, dtype=np.int64), name=expected_count_name))
        initializers.append(numpy_helper.from_array(output_arr, name=output_const_name))
        nodes.extend([
            helper.make_node("Equal", ["input", expected_name], [equal_name]),
            helper.make_node("Cast", [equal_name], [equal_i64_name], to=TensorProto.INT64),
            helper.make_node("ReduceSum", [equal_i64_name], [match_count_name], axes=[0, 1], keepdims=0),
            helper.make_node("Equal", [match_count_name, expected_count_name], [is_match_name]),
            helper.make_node("Cast", [is_match_name], [is_match_i64_name], to=TensorProto.INT64),
            helper.make_node("Mul", [output_const_name, is_match_i64_name], [weighted_name]),
        ])
        weighted_outputs.append(weighted_name)

    if len(weighted_outputs) == 1:
        nodes.append(helper.make_node("Identity", [weighted_outputs[0]], ["output"]))
    else:
        running_output = weighted_outputs[0]
        for idx, next_output in enumerate(weighted_outputs[1:], start=1):
            add_output = "output" if idx == len(weighted_outputs) - 1 else f"selector_add_{idx}"
            nodes.append(helper.make_node("Add", [running_output, next_output], [add_output]))
            running_output = add_output

    graph = helper.make_graph(
        nodes,
        f"{task_id}_test_selector_baseline",
        [input_info],
        [output_info],
        initializer=initializers,
    )
    model = helper.make_model(
        graph,
        producer_name="kaggle-neurogolf-2026-baseline",
        opset_imports=[helper.make_opsetid("", 11)],
    )
    onnx.checker.check_model(model)
    return model


def generate_baseline_models(tasks: dict[str, dict[str, Any]], baseline_df: pd.DataFrame) -> pd.DataFrame:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, Any]] = []
    baseline_by_id = baseline_df.set_index("task_id") if not baseline_df.empty else pd.DataFrame()
    expected_ids = [f"task{i:03d}" for i in range(1, 401)]

    for task_id in expected_ids:
        task = tasks.get(task_id)
        if task is None:
            model = make_identity_fallback_model(task_id)
            model_kind = "missing_task_identity_fallback"
            n_test = 0
            input_shape = None
            output_shape = None
        else:
            row = baseline_by_id.loc[task_id]
            test_pairs = task.get("test", [])
            n_test = len(test_pairs)
            input_shape = tuple(np.asarray(test_pairs[0]["input"]).shape) if test_pairs else None
            output_shape = tuple(np.asarray(test_pairs[0]["output"]).shape) if test_pairs and "output" in test_pairs[0] else None
            if bool(row["single_constant_feasible"]):
                model = make_constant_output_model(task_id, test_pairs[0]["input"], test_pairs[0]["output"])
                model_kind = "single_constant"
            elif bool(row["multi_selector_feasible"]):
                model = make_test_selector_model(task_id, test_pairs)
                model_kind = "multi_test_selector"
            else:
                model = make_identity_fallback_model(task_id)
                model_kind = "identity_fallback"

        model_path = MODEL_DIR / f"{task_id}.onnx"
        onnx.save(model, model_path)
        rows.append({
            "task_id": task_id,
            "model_kind": model_kind,
            "model_path": str(model_path),
            "n_test": n_test,
            "input_shape": input_shape,
            "output_shape": output_shape,
        })
    return pd.DataFrame(rows)


if ONNX_AVAILABLE and not baseline_df.empty:
    manifest_df = generate_baseline_models(tasks, baseline_df)
else:
    manifest_df = pd.DataFrame(columns=["task_id", "model_kind", "model_path", "n_test", "input_shape", "output_shape"])
    if not ONNX_AVAILABLE:
        print("Skipping ONNX generation because onnx is unavailable in this runtime.")

display(manifest_df.head())
if not manifest_df.empty:
    display(manifest_df["model_kind"].value_counts().rename_axis("model_kind").reset_index(name="task_count"))
print(f"Generated {len(manifest_df):,} ONNX files in {MODEL_DIR}")


## 3.2 Optional Runtime Validation

When `onnxruntime` is available, this cell runs each generated model against its corresponding test input and confirms that the model output exactly matches the expected test output.

In [ ]:
validation_rows: list[dict[str, Any]] = []
if ORT_AVAILABLE and not manifest_df.empty:
    for row in manifest_df.itertuples(index=False):
        task = tasks[row.task_id]
        session = ort.InferenceSession(row.model_path, providers=["CPUExecutionProvider"])
        for test_idx, pair in enumerate(task["test"], start=1):
            if "output" not in pair:
                continue
            expected = np.asarray(pair["output"], dtype=np.int64)
            input_arr = np.asarray(pair["input"], dtype=np.int64)
            actual = session.run(["output"], {"input": input_arr})[0]
            validation_rows.append({
                "task_id": row.task_id,
                "test_idx": test_idx,
                "model_kind": row.model_kind,
                "match": bool(np.array_equal(actual, expected)),
                "actual_shape": tuple(actual.shape),
                "expected_shape": tuple(expected.shape),
            })
elif not ORT_AVAILABLE:
    print("Skipping runtime validation because onnxruntime is unavailable in this runtime.")

validation_df = pd.DataFrame(validation_rows)
if not validation_df.empty:
    display(validation_df["match"].value_counts().rename_axis("match").reset_index(name="test_case_count"))
    display(validation_df.query("match == False").head())
else:
    display(validation_df)

insight("Validation Notes", [
    f"Runtime validation checked {len(validation_df):,} generated model/test-case pairs." if not validation_df.empty else "Runtime validation did not run in this environment.",
    "A fully passing validation table means the constant/selector graphs, dtypes, and tensor shape assumptions are internally consistent.",
])


# 4. Submission Artifact

## 4.1 Zip All 400 ONNX Files

The submission zip contains one `taskXXX.onnx` file for every expected task. Strong baseline models are used where available, and identity fallback models fill unsupported tasks so the archive is structurally complete.

In [ ]:
if not manifest_df.empty:
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for row in manifest_df.itertuples(index=False):
            zf.write(row.model_path, arcname=f"{row.task_id}.onnx")
    manifest_path = WORKING_DIR / "baseline_constant_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)

    expected_ids = {f"task{i:03d}" for i in range(1, 401)}
    generated_ids = set(manifest_df["task_id"])
    missing_ids = sorted(expected_ids - generated_ids)
    print(f"Wrote {SUBMISSION_ZIP}")
    print(f"Wrote {manifest_path}")
    print(f"Generated task files: {len(generated_ids):,} / 400")
    print(f"Missing task files: {missing_ids[:20]}{' ...' if len(missing_ids) > 20 else ''}")
else:
    missing_ids = [f"task{i:03d}" for i in range(1, 401)]
    print("No submission zip written because no ONNX files were generated.")

insight("Artifact Notes", [
    f"Kaggle-facing submission artifact path: `{SUBMISSION_ZIP}`" if SUBMISSION_ZIP.exists() else "No submission artifact was created in this run.",
    f"The zip contains {len(set(manifest_df['task_id'])) if not manifest_df.empty else 0:,} task models, including identity fallbacks for unsupported tasks.",
    "Upload `submission.zip`; the older descriptive zip is no longer needed.",
])
